In [4]:
import time
import serial


def calculate_crc16(data: bytes) -> bytes:
    """Вычисляет контрольную сумму CRC16 для протокола Modbus (Modbus CRC)."""
    crc = 0xFFFF
    for pos in data:
        crc ^= pos
        for _ in range(8):
            if (crc & 0x0001) != 0:
                crc >>= 1
                crc ^= 0xA001
            else:
                crc >>= 1
    # Возвращает 2 байта в формате Little Endian (младший байт вперед)
    return bytes([crc & 0xFF, (crc >> 8) & 0xFF])


# --- НАСТРОЙКИ ПОДКЛЮЧЕНИЯ ---
PORT_NAME = r"\\.\COM3"
BAUDRATE = 9600
STOPBITS = serial.STOPBITS_TWO  # Поставьте STOPBITS_ONE, если в приборе 1 стоп-бит

SLAVE_ID = 18  # Адрес прибора (0x12)
FUNCTION_CODE = 3  # Функция 03 — Read Holding Registers
REGISTER_ADDRESS = 201  # Регистр, который хотим прочитать
COUNT = 1  # Количество регистров для чтения

# --- СБОРКА ПАКЕТА ЗАПРОСА ---
# Разбираем адрес регистра и количество на байты (старший, затем младший)
reg_hi, reg_lo = (REGISTER_ADDRESS >> 8) & 0xFF, REGISTER_ADDRESS & 0xFF
count_hi, count_lo = (COUNT >> 8) & 0xFF, COUNT & 0xFF

# Собираем заголовок пакета
packet_without_crc = bytes(
    [SLAVE_ID, FUNCTION_CODE, reg_hi, reg_lo, count_hi, count_lo]
)

# Считаем и добавляем CRC16
request = packet_without_crc + calculate_crc16(packet_without_crc)

print(f"Отправляем запрос (HEX): {request.hex().upper()}")

# --- ОТПРАВКА И ПРИЕМ ПО СЕТИ ---
try:
    ser = serial.Serial(
        port=PORT_NAME,
        baudrate=BAUDRATE,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE,
        stopbits=STOPBITS,
        timeout=1.0,
    )

    ser.reset_input_buffer()
    ser.reset_output_buffer()

    # Отправляем 8 байт в линию RS-485
    ser.write(request)

    # Небольшая пауза, чтобы прибор успел ответить
    time.sleep(0.1)

    # Проверяем ответ
    if ser.in_waiting > 0:
        # Для чтения 1 регистра ответ всегда должен быть длиной 7 байт
        response = ser.read(ser.in_waiting)
        print(f"Получен ответ    (HEX): {response.hex().upper()}")

        # Проверяем контрольную сумму ответа
        received_crc = response[-2:]
        calculated_crc = calculate_crc16(response[:-2])

        if received_crc == calculated_crc:
            # Парсим значение (данные ответа начинаются с 3-го байта)
            # Байт 0: Slave ID, Байт 1: Fn Code, Байт 2: Кол-во байт данных (0x02)
            # Байт 3-4: Само значение регистра (Старший байт, Младший байт)
            val_high = response[3]
            val_low = response[4]
            register_value = (val_high << 8) | val_low

            print("-" * 30)
            print(f"УСПЕХ! Значение регистра 201: {register_value}")
            print("-" * 30)
        else:
            print("Ошибка: Контрольная сумма ответа не совпала (помехи).")
    else:
        print("Ошибка: Устройство №18 не ответило (Таймаут).")

except Exception as e:
    print(f"Системная ошибка порта: {e}")
finally:
    if "ser" in locals() and ser.is_open:
        ser.close()



Отправляем запрос (HEX): 120300C900015697
Получен ответ    (HEX): 120302169A00007495
------------------------------
УСПЕХ! Значение регистра 201: 5786
------------------------------
